***Task1***

In [33]:
import sqlite3
import pandas as pd

In [34]:
conn = sqlite3.connect('library.db.html')

1. How much is each member borrowing?

In [35]:
q1_sql = """
SELECT
    m.member_id,
    m.first_name,
    m.last_name,
    COUNT(c.checkout_id) AS total_checkouts
FROM members m
LEFT JOIN checkouts c ON m.member_id = c.member_id
GROUP BY m.member_id;
"""
df_q1 = pd.read_sql_query(q1_sql, conn)
df_q1.head()

,member_id,first_name,last_name,total_checkouts
0,1001,Salma,Ibrahim,1
1,1002,Fares,Saleh,2
2,1003,Bassel,Hegazy,9
3,1004,Fares,Wahba,0
4,1005,Youssef,Halim,3


2. Which books match a chosen author pattern?

In [36]:
q2_sql = """
SELECT
    book_id,
    title,
    author
FROM books
WHERE author LIKE '%Smith%';
"""
df_q2 = pd.read_sql_query(q2_sql, conn)
df_q2.head()

,book_id,title,author


3. What are the most popular books?

In [37]:
q3_sql = """
SELECT
    b.title,
    COUNT(c.checkout_id) AS times_borrowed
FROM checkouts c
JOIN books b ON c.book_id = b.book_id
GROUP BY b.book_id
ORDER BY times_borrowed DESC
LIMIT 5;
"""
df_q3 = pd.read_sql_query(q3_sql, conn)
df_q3.head()

,title,times_borrowed
0,The Silver Kite,57
1,Fossils and Fireflies,55
2,Circuits for Beginners,46
3,Kites Over Cairo,38
4,Storms and Sailboats,25


4. Who are the most active readers?

In [38]:
q4_sql = """
SELECT
    m.member_id,
    m.first_name,
    m.last_name,
    COUNT(c.checkout_id) AS total_borrowed
FROM members m
JOIN checkouts c ON m.member_id = c.member_id
GROUP BY m.member_id
ORDER BY total_borrowed DESC
LIMIT 10;
"""
df_q4 = pd.read_sql_query(q4_sql, conn)
df_q4.head()

,member_id,first_name,last_name,total_borrowed
0,1034,Aya,Wahba,25
1,1044,Sherif,Saleh,21
2,1008,Ziad,Saleh,19
3,1027,Mostafa,Fouad,18
4,1010,Nour,Nabil,18


5. What does a neighborhood's activity look like further back in time?

In [39]:
q5_sql = """
SELECT
    c.checkout_id,
    m.first_name,
    m.last_name,
    m.neighborhood,
    c.checkout_date
FROM checkouts c
JOIN members m ON c.member_id = m.member_id
WHERE m.neighborhood = 'Heliopolis'
ORDER BY c.checkout_date DESC
LIMIT 100 OFFSET 10;
"""
df_q5 = pd.read_sql_query(q5_sql, conn)
df_q5.head()

,checkout_id,first_name,last_name,neighborhood,checkout_date
0,9210,Sherif,Saleh,Heliopolis,2025-09-11
1,9230,Fares,Adel,Heliopolis,2025-09-10
2,9261,Sara,Rashad,Heliopolis,2025-09-02
3,9225,Lina,Halim,Heliopolis,2025-08-06
4,9250,Sherif,Saleh,Heliopolis,2025-08-05


In [40]:
members_df = pd.read_sql_query("SELECT * FROM members", conn)
checkouts_df = pd.read_sql_query("SELECT * FROM checkouts", conn)
df = pd.merge(checkouts_df, members_df, on='member_id', how='left')
df.head()

,checkout_id,member_id,book_id,checkout_date,return_date,first_name,last_name,grade,neighborhood,membership_status,join_date
0,9263,1047,517,2024-10-21,2024-11-07,Sara,Rashad,NaN,Heliopolis,Inactive,2024-06-25
1,9340,1072,513,2025-08-24,2025-09-01,Seif,Zaki,9.0,Zamalek,Active,2025-10-21
2,9231,1053,523,2024-02-04,2024-02-16,Adam,Shafik,9.0,Heliopolis,Active,2024-01-03
3,9129,1032,513,2025-06-21,2025-06-29,Nada,Zaki,7.0,Nasr City,Active,2025-10-19
4,9370,1079,511,2025-11-11,2025-12-03,Rana,Osman,8.0,Shubra,Active,2024-10-27


In [41]:
books_db = pd.read_sql_query("SELECT * FROM books", conn)
books_json = pd.read_json("books.json")
books_all = pd.merge(books_db, books_json, on='book_id', how='left')
df = pd.merge(df, books_all, on='book_id', how='left')
df['source'] = 'database'
df.head()

,checkout_id,member_id,book_id,checkout_date,return_date,first_name,last_name,grade,neighborhood,membership_status,join_date,title,author,genre,pages,publication_year,publisher,source
0,9263,1047,517,2024-10-21,2024-11-07,Sara,Rashad,NaN,Heliopolis,Inactive,2024-06-25,Shadows on the Corniche,Hani Nagati,Mystery,338,2015.0,Delta House,database
1,9340,1072,513,2025-08-24,2025-09-01,Seif,Zaki,9.0,Zamalek,Active,2025-10-21,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books,database
2,9231,1053,523,2024-02-04,2024-02-16,Adam,Shafik,9.0,Heliopolis,Active,2024-01-03,Footsteps in the Dust,Laila Shokry,Historical,276,2018.0,Oasis Books,database
3,9129,1032,513,2025-06-21,2025-06-29,Nada,Zaki,7.0,Nasr City,Active,2025-10-19,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books,database
4,9370,1079,511,2025-11-11,2025-12-03,Rana,Osman,8.0,Shubra,Active,2024-10-27,Winter in Alexandria,Farida Anwar,Historical,117,2016.0,Nile Press,database


In [42]:
html_tables = pd.read_html("summer_checkouts.html.html")
html_df = html_tables[0]

html_df = html_df.rename(columns={
    'Member ID': 'member_id',
    'Book ID': 'book_id',
    'Checkout Date': 'checkout_date'
})

html_merged = pd.merge(html_df, members_df, on='member_id', how='left')
html_merged = pd.merge(html_merged, books_all, on='book_id', how='left')

html_merged['checkout_id'] = None
html_merged['return_date'] = None
html_merged['source'] = 'html_scrape'

cols_order = df.columns
html_merged = html_merged[cols_order]

final_dataset = pd.concat([df, html_merged], ignore_index=True)
final_dataset.head()

,checkout_id,member_id,book_id,checkout_date,return_date,first_name,last_name,grade,neighborhood,membership_status,join_date,title,author,genre,pages,publication_year,publisher,source
0,9263,1047,517,2024-10-21,2024-11-07,Sara,Rashad,NaN,Heliopolis,Inactive,2024-06-25,Shadows on the Corniche,Hani Nagati,Mystery,338,2015.0,Delta House,database
1,9340,1072,513,2025-08-24,2025-09-01,Seif,Zaki,9.0,Zamalek,Active,2025-10-21,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books,database
2,9231,1053,523,2024-02-04,2024-02-16,Adam,Shafik,9.0,Heliopolis,Active,2024-01-03,Footsteps in the Dust,Laila Shokry,Historical,276,2018.0,Oasis Books,database
3,9129,1032,513,2025-06-21,2025-06-29,Nada,Zaki,7.0,Nasr City,Active,2025-10-19,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books,database
4,9370,1079,511,2025-11-11,2025-12-03,Rana,Osman,8.0,Shubra,Active,2024-10-27,Winter in Alexandria,Farida Anwar,Historical,117,2016.0,Nile Press,database


In [43]:
final_dataset.to_csv("combined_library_dataset.csv", index=False)

***Task2***

In [44]:
import pandas as pd

In [45]:
df = pd.read_csv("combined_library_dataset.csv")

***4problems***

***Frist:Problem 1***

In [46]:
df.isnull().sum()

,0
checkout_id,26
member_id,0
book_id,0
checkout_date,0
return_date,91
first_name,5
last_name,5
grade,41
neighborhood,5
membership_status,5


***Problem 2***

In [47]:
df.duplicated().sum()

np.int64(8)

***Problem 3***

In [48]:
df['neighborhood'].unique()
df['neighborhood'].value_counts(dropna=False)

,count
neighborhood,
Nasr City,103
Maadi,96
Heliopolis,88
Zamalek,60
Shubra,34
Maadi,19
zamalek,10
NaN,5
NASR CITY,1


***Problem 4***

In [49]:
df[df['first_name'].isna()].shape[0]

5

***Answer problem1***

In [50]:
df['genre'].isna().sum()

df['genre'] = df['genre'].replace(r'^\s*$', None, regex=True)
df['genre'] = df['genre'].fillna('Unknown')

df['genre'].isna().sum()

df[['book_id', 'title', 'genre']].head(10)

,book_id,title,genre
0,517,Shadows on the Corniche,Mystery
1,513,Circuits for Beginners,Science
2,523,Footsteps in the Dust,Historical
3,513,Circuits for Beginners,Science
4,511,Winter in Alexandria,Historical
5,528,The Glass Beehive,Nature
6,513,Circuits for Beginners,Science
7,501,The Silver Kite,Adventure
8,506,The Paper Boat Club,Friendship
9,506,The Paper Boat Club,Friendship


***Aswer problem 2***

In [51]:
before_dup = len(df)
df = df.drop_duplicates()
after_dup = len(df)
print(f"Before dropping duplicates: {before_dup}")
print(f"After dropping duplicates: {after_dup}")
print(f"Number of duplicates dropped: {before_dup - after_dup}")

Before dropping duplicates: 417
After dropping duplicates: 409
Number of duplicates dropped: 8


***Answer Problem 3***

In [52]:
df_clean = df[df['first_name'].notna()].copy()

df_clean['neighborhood'] = df_clean['neighborhood'].str.strip().str.title()

print(df_clean['neighborhood'].unique())

['Heliopolis' 'Zamalek' 'Nasr City' 'Shubra' 'Maadi']


***Answer Problem 4***

In [53]:
df_clean = df[df['first_name'].notna()].copy()
df_clean['first_name'] = df_clean['first_name'].str.title()
df_clean['last_name'] = df_clean['last_name'].str.title()
df_clean

,checkout_id,member_id,book_id,checkout_date,return_date,first_name,last_name,grade,neighborhood,membership_status,join_date,title,author,genre,pages,publication_year,publisher,source
0,9263.0,1047,517,2024-10-21,2024-11-07,Sara,Rashad,NaN,Heliopolis,Inactive,2024-06-25,Shadows on the Corniche,Hani Nagati,Mystery,338,2015.0,Delta House,database
1,9340.0,1072,513,2025-08-24,2025-09-01,Seif,Zaki,9.0,Zamalek,Active,2025-10-21,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books,database
2,9231.0,1053,523,2024-02-04,2024-02-16,Adam,Shafik,9.0,Heliopolis,Active,2024-01-03,Footsteps in the Dust,Laila Shokry,Historical,276,2018.0,Oasis Books,database
3,9129.0,1032,513,2025-06-21,2025-06-29,Nada,Zaki,7.0,Nasr City,Active,2025-10-19,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books,database
4,9370.0,1079,511,2025-11-11,2025-12-03,Rana,Osman,8.0,Shubra,Active,2024-10-27,Winter in Alexandria,Farida Anwar,Historical,117,2016.0,Nile Press,database
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
411,NaN,1073,530,2025-07-07,NaN,Yassin,Kamel,7.0,Zamalek,Inactive,2025-08-26,The Sandstone Key,Sara Tantawy,Adventure,208,2012.0,Oasis Books,html_scrape
412,NaN,1003,501,2025-07-08,NaN,Bassel,Hegazy,6.0,Maadi,Active,2025-04-23,The Silver Kite,Amina Darwish,Adventure,128,2017.0,Nile Press,html_scrape
413,NaN,1017,507,2025-07-11,NaN,Adam,Badr,9.0,Maadi,Active,2025-07-12,Fossils and Fireflies,Dalia Serry,Science,160,2024.0,Nile Press,html_scrape
414,NaN,1061,504,2025-07-06,NaN,Ziad,Fahmy,9.0,zamalek,Inactive,2023-04-27,Rooftop Astronomers,Adel Roushdy,Science,319,2009.0,Cairo Young Readers,html_scrape


In [54]:
df_clean.to_csv("task2_cleaned_data.csv", index=False)

***Task 3***

In [55]:
fairness_df = df_clean.groupby('neighborhood').agg(
    total_members=('member_id', 'nunique'),
    total_checkouts=('checkout_id', 'count')
).reset_index()
print("--- Neighborhood Fairness Summary ---")
print(fairness_df)

--- Neighborhood Fairness Summary ---
  neighborhood  total_members  total_checkouts
0   HELIOPOLIS              1                1
1   Heliopolis             12               83
2        Maadi             18               88
3       Maadi               2               18
4    NASR CITY              1                1
5    Nasr City             15               96
6       Shubra              5               34
7      Zamalek             10               54
8      zamalek              1                8
